# 03 - Edits and deletions

The two layers that only repeated revisits can produce. A one-pass corpus records
what a channel had published at the moment it was scraped; it cannot know that a
message was later rewritten, and it cannot know that one disappeared. TGDataset2
revisits, so it can.

**Figures produced here**

| Output | Thesis figure |
|---|---|
| `edit_automatic_vs_genuine` | what the `edit_date` flag actually counts |
| `fig_5_8_edit_taxonomy` | edit-type taxonomy over the diffable pairs |
| `deletion_rates_by_role` | deletion rate by message role |
| `deleted_comment_root_status` | why a comment is deleted |
| `fig_6_11_moderation_vs_purge` | toxicity of independently moderated vs purged comments |

**Table produced here**: deletion rates by role.

A framing point that applies to everything below: **deletion here is detection,
not deletion.** The corpus records that a message present on one visit was absent
on a later one, with `deleted_at` being the detection time, not the moment the
author pressed delete. Everything is therefore a lower bound, and it only covers
the window in which revisits were running.

Sections marked *offline* read a CSV from `../data/`; the rest need
`TGDATASET2_DB_URL`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from _shared import DATA, FIGURES, csv, q, setup_plots

setup_plots()

## 1. The edit flag lies (*offline*)

Telegram sets `edit_date` on a message when it is edited, and that flag is the
obvious proxy for "someone changed this". It is set on roughly a third of all
messages in the corpus, which would make Telegram an extraordinarily heavily
edited medium.

It is not a human-edit signal. Telegram sets `edit_date` for its own reasons: when
a media album finishes uploading, when a link preview resolves, when an
auto-updating tally moves. Those land within seconds of posting and nobody
touched anything.

The test is latency crossed with content kind. If the early spike were human
editing, it would look the same for text-only posts as for media posts. It does
not: the sub-ten-second bucket is dominated by media and link previews, while
text-only edits skew late, which is what a person rereading their own post looks
like.

The `TABLESAMPLE SYSTEM (0.3)` clause is a 0.3 % page-level sample. The
alternative is a full scan of a table with no index on `edit_date`, and the shape
of a three-bucket latency split does not need every row.

In [ ]:
SQL_EDIT_LATENCY = """
    SELECT CASE
             WHEN media_type IS NULL     THEN 'text-only'
             WHEN media_type = 'webpage' THEN 'webpage (link preview)'
             ELSE 'media (photo/video/...)'
           END AS kind,
           count(*) FILTER (WHERE gap < 10)                AS instant_lt10s,
           count(*) FILTER (WHERE gap >= 10 AND gap < 300) AS soon_10s_5m,
           count(*) FILTER (WHERE gap >= 300)              AS later_gt5m,
           count(*)                                        AS total
    FROM (SELECT media_type, EXTRACT(EPOCH FROM (edit_date - date)) AS gap
          FROM messages TABLESAMPLE SYSTEM (0.3)
          WHERE edit_date IS NOT NULL AND edit_date >= date) s
    GROUP BY kind
    ORDER BY total DESC
"""

comp = csv("edit_automatic_vs_genuine.csv")
tot = int(comp["total"].sum())
inst = int(comp["instant_lt10s"].sum())
media_inst = int(comp.loc[comp["kind"] != "text-only", "instant_lt10s"].sum())
print(f"flagged-edit sample n={tot:,}")
print(f"land within 10 s of posting: {inst:,} ({inst / tot:.0%}), too fast to be human")
print(f"  of those, media or webpage: {media_inst / inst:.0%} (automatic finalisation)")

fig, ax = plt.subplots(figsize=(5.48, 2.19))
b = comp.set_index("kind")[["instant_lt10s", "soon_10s_5m", "later_gt5m"]]
b.plot.barh(stacked=True, ax=ax, color=["#d4584f", "#e1a33a", "#5b8ef0"])
ax.set_xlabel("flagged edits (sample)")
ax.set_title("What the edit_date flag counts, by latency")
ax.legend(["<10 s (automatic)", "10 s to 5 min", ">5 min (likely human)"], fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "edit_automatic_vs_genuine.png", bbox_inches="tight")
fig.savefig(FIGURES / "edit_automatic_vs_genuine.pdf", bbox_inches="tight")
plt.show()
print("edit_date conflates Telegram's automatic media and preview edits with human "
      "edits. The clean human-edit signal is the text-bearing, slower tail.")

## 2. What an edit actually does

The corrective for the flag is the `message_edits` archive: when a revisit finds a
message whose text differs from the stored version, the scraper keeps the prior
text. That gives before/after pairs, which is what no flag-only corpus has.

Of the archived edits, the overwhelming majority leave the text byte-identical:
the change was in media, markup, or entities. The pairs where the text genuinely
changed are the working set.

The query is small and keyed, so it is cheap even on the full corpus.

In [ ]:
SQL_EDIT_PAIRS = """
    SELECT e.channel_id, e.message_id,
           e.text AS before_text,
           m.text AS after_text,
           m.date,
           length(m.text) - length(e.text) AS char_delta
    FROM message_edits e
    JOIN messages m USING (channel_id, message_id)
    WHERE e.text IS DISTINCT FROM m.text
      AND e.text IS NOT NULL AND m.text IS NOT NULL
      AND length(e.text) > 0
"""

edit_pairs = q(SQL_EDIT_PAIRS)
# message_edits can hold more than one archived version per message, so a handful
# of keys repeat; the paired design is one row per (channel_id, message_id).
edit_pairs = edit_pairs.drop_duplicates(["channel_id", "message_id"], keep="first")
print(f"text-changing edit pairs: {len(edit_pairs):,}")

### The taxonomy, and how the labels were made

Each text-changing pair carries exactly one primary label: the *dominant* kind of
change. The taxonomy is fixed at nine types in two groups.

**Non-substantive**

- `cosmetic`: spelling, spacing, punctuation, emoji, markup or hashtag only;
  meaning unchanged.
- `mechanical_counter`: only an auto-updating tally changed (poll votes,
  reaction, comment or view counts).

**Substantive editorial**

- `factual_correction`: a number, date, name, place or fact changed.
- `addition`: new substantive content, links, promotional material or a status
  note inserted.
- `removal`: substantive content cut or retracted.
- `tone_rewrite`: rephrased without adding or removing facts.
- `compliance_annotation`: a state- or platform-mandated label added or removed
  (a "foreign agent" notice, a terms-of-service disclaimer).
- `multi`: two or more co-equal substantive changes in one edit.
- `other`: none of the above.

**How the labels were produced.** The 2,356 pairs were labelled in one pass by an
LLM (Claude Sonnet 4.6) against the taxonomy above, in batches of 25 pairs per
call, each call receiving the archived before-text and the current after-text and
returning one label plus a short note per pair. The full prompt is reproduced
verbatim in the thesis, `thesis/chapters/appendix_edit_prompt.tex`, so the
labelling is reproducible and auditable rather than a black box.

Two things follow from that, and the thesis says both. The substantive versus
non-substantive split is a *derived* boundary: it comes from grouping the nine
labels, with `cosmetic` and `mechanical_counter` counted as non-substantive. And
the labels are model output applied cross-lingually without task-specific
ground-truth validation on this corpus, so they carry direction and contrast
rather than calibrated absolute levels. A 1,000-pair subsample was independently
relabelled by a second model as a consistency check.

**The label file is not shipped.** Each label carries a free-text note naming the
concrete change, and those notes quote message content and occasionally personal
names. Publishing them would republish user content the corpus does not have the
right to redistribute, so this section needs both the database and a local
labelling run.

In [ ]:
LABELS_PATH = DATA / "edit_labels.jsonl"   # not shipped; see the note above

NON_SUBSTANTIVE = {"cosmetic", "mechanical_counter"}
TAXONOMY_ORDER = ["cosmetic", "mechanical_counter", "factual_correction", "addition",
                  "removal", "tone_rewrite", "compliance_annotation", "multi", "other"]

labels = (pd.read_json(LABELS_PATH, lines=True)
            .drop_duplicates(["channel_id", "message_id"], keep="last"))
edits = labels.merge(
    edit_pairs[["channel_id", "message_id", "before_text", "after_text", "char_delta"]],
    on=["channel_id", "message_id"], how="left")
edits["substantive"] = ~edits["label"].isin(NON_SUBSTANTIVE)

dist = edits["label"].value_counts().reindex(TAXONOMY_ORDER).fillna(0).astype(int)
n = len(edits)
subst = int(edits["substantive"].sum())
print(f"labelled pairs: {n:,}")
print(f"substantive editorial: {subst:,} ({subst / n:.0%})   "
      f"non-substantive: {n - subst:,} ({(n - subst) / n:.0%})")
for lab in TAXONOMY_ORDER:
    c = int(dist[lab])
    mark = "   " if lab in NON_SUBSTANTIVE else " * "
    print(f"  {mark}{lab:<22}{c:>5}  ({100 * c / n:4.1f}%)")

In [ ]:
order = dist[dist > 0]
colors = ["#bbbbbb" if lab in NON_SUBSTANTIVE else "#4c72b0" for lab in order.index]

fig, ax = plt.subplots(figsize=(5.48, 3.08))
ax.barh(range(len(order)), order.values, color=colors)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order.index)
ax.invert_yaxis()
top = int(max(order.values))
for i, v in enumerate(order.values):
    ax.text(v + top * 0.01, i, f"{v:,} ({100 * v / n:.0f}%)", va="center", fontsize=7)
ax.set_xlim(0, top * 1.18)
ax.set_xlabel("text-changing edits")
ax.set_title(f"Edit-type taxonomy (n={n:,})\n"
             f"grey = non-substantive (cosmetic / counter)   "
             f"blue = substantive editorial")
fig.tight_layout()
fig.savefig(FIGURES / "fig_5_8_edit_taxonomy.png", dpi=150)
fig.savefig(FIGURES / "fig_5_8_edit_taxonomy.pdf")
plt.show()

## 3. Deletion rates by role (*offline*)

Three roles of message, with very different deletion behaviour:

- **broadcast post**: a message in a channel that is not a discussion group.
- **comment**: a message in a linked discussion group that carries a reply
  pointer.
- **auto-forward head**: the channel's own mirrored copy of a post inside its
  linked group. Identified by three conditions together, which is what
  distinguishes a genuine mirror from a member manually re-forwarding the post:
  it is a forward, it comes from the parent channel, and its author *is* the
  parent channel.

Denominators and deleted counts come out of one pass, so the rates are internally
consistent.

In [ ]:
SQL_DELETION_OVERVIEW = """
    SELECT
      count(*)                                                                                AS total_msgs,
      count(*) FILTER (WHERE c.linked_to_channel_id IS NULL)                                  AS broadcast_msgs,
      count(*) FILTER (WHERE c.linked_to_channel_id IS NOT NULL)                              AS linked_group_msgs,
      count(*) FILTER (WHERE c.linked_to_channel_id IS NOT NULL
                        AND (m.reply_to_top_id IS NOT NULL OR m.reply_to_msg_id IS NOT NULL)) AS comments,
      count(*) FILTER (WHERE c.linked_to_channel_id IS NOT NULL
                        AND m.reply_to_top_id IS NULL AND m.reply_to_msg_id IS NULL
                        AND m.is_forwarded AND m.fwd_from_channel_id = c.linked_to_channel_id
                        AND m.author_id = c.linked_to_channel_id)                             AS heads,
      count(*) FILTER (WHERE m.is_deleted)                                                    AS del_total,
      count(*) FILTER (WHERE m.is_deleted AND c.linked_to_channel_id IS NULL)                 AS del_broadcast,
      count(*) FILTER (WHERE m.is_deleted AND c.linked_to_channel_id IS NOT NULL
                        AND (m.reply_to_top_id IS NOT NULL OR m.reply_to_msg_id IS NOT NULL)) AS del_comments,
      count(*) FILTER (WHERE m.is_deleted AND c.linked_to_channel_id IS NOT NULL
                        AND m.reply_to_top_id IS NULL AND m.reply_to_msg_id IS NULL
                        AND m.is_forwarded AND m.fwd_from_channel_id = c.linked_to_channel_id
                        AND m.author_id = c.linked_to_channel_id)                             AS del_heads
    FROM messages m JOIN channels c ON c.channel_id = m.channel_id
"""

rates = csv("deletion_rates_by_role.csv")
display(rates)

fig, ax = plt.subplots(figsize=(5.48, 3.13))
bars = ax.bar(rates["role"], rates["del_rate"] * 100,
              color=["#d4584f", "#5b8ef0", "#7aa86f"])
ax.set_ylabel("deletion rate (% of role's messages)")
ax.set_title("Deletion rate by message role")
for b, cnt in zip(bars, rates["deleted"]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
            f"{b.get_height():.3f}%\n({cnt:,})", ha="center", va="bottom", fontsize=8)
ax.margins(y=0.18)
fig.tight_layout()
fig.savefig(FIGURES / "deletion_rates_by_role.png", bbox_inches="tight")
fig.savefig(FIGURES / "deletion_rates_by_role.pdf", bbox_inches="tight")
plt.show()
print("Heads are deleted far less often than posts or comments, a first hint that "
      "comment deletion is not simply a downstream effect of post deletion.")

## 4. Why a comment disappears (*offline*)

A deleted comment has two very different possible causes, and telling them apart
is the point of this section. Either someone removed *that comment* (moderation,
or the author deleting their own message), or the whole thread went, taking every
comment under it with it (a purge).

The test looks up from each deleted comment to its thread root and asks what
happened to the root. Five outcomes, in decreasing order of what they let you
conclude:

- **root alive or uncollected**: the comment went on its own. This is the
  independent-moderation bucket, and it is an upper bound: a root that was simply
  not rescanned yet also lands here.
- **root = deleted parent-head**: the mirrored copy of the post is gone, so the
  whole post thread was wiped.
- **root = deleted native group message**, **root = deleted forwarded-in post**,
  **root = another deleted comment**: variations on the same purge story, at
  different points in the tree.

In [ ]:
SQL_DELETED_SET = """
    SELECT m.channel_id, m.message_id, left(m.text, 200) AS text,
           char_length(m.text) AS textlen, m.reply_to_top_id, m.reply_to_msg_id,
           m.is_forwarded, m.fwd_from_channel_id, m.views,
           c.linked_to_channel_id AS parent_id
    FROM messages m JOIN channels c ON c.channel_id = m.channel_id
    WHERE m.is_deleted
"""
# Each deleted comment's root is looked up in the deleted key set itself:
#
#   root_id      = reply_to_top_id, falling back to reply_to_msg_id
#   root_deleted = (channel_id, root_id) appears in the deleted set
#
# and a deleted root is then classified by its own columns: a forward from the
# parent channel is the post's own thread head; a forward from elsewhere is a
# forwarded-in post; a row carrying its own reply pointer is another comment;
# anything left is a plain group message.

out = csv("deleted_comment_root_status.csv", index_col=0)
display(out)

fig, ax = plt.subplots(figsize=(5.48, 2.58))
colors = ["#9aa0a6", "#d4584f", "#e1a33a", "#c77dff", "#6b8e23"]
order = out.sort_values("comments", ascending=True)
ax.barh(order.index, order["share"] * 100, color=colors[:len(order)])
ax.set_xlabel("% of deleted comments")
ax.set_title("Why is a comment deleted? (status of its thread root)")
for i, (_, row) in enumerate(order.iterrows()):
    ax.text(row["share"] * 100, i,
            f" {row['share'] * 100:.1f}%  ({int(row['comments']):,})",
            va="center", fontsize=8)
ax.margins(x=0.20)
fig.tight_layout()
fig.savefig(FIGURES / "deleted_comment_root_status.png", bbox_inches="tight")
fig.savefig(FIGURES / "deleted_comment_root_status.pdf", bbox_inches="tight")
plt.show()

## 5. Does moderation remove worse content than a purge does? (*offline*)

If the independent-moderation bucket really is moderation, its contents should
look different from the purge bucket: a moderator removes a comment *because of
what it says*, whereas a thread wipe takes everything under the post regardless.

The contrast is measured on a capped random sample from each bucket, scored with
the `textdetox` XLM-R toxicity classifier and a few cheap surface features.
Sampling is capped per bucket so the two are balanced, and seeded so the draw is
reproducible.

Two caveats, both material. The classifier is applied cross-lingually on a
multilingual corpus without task-specific validation, so it gives direction and
contrast, not calibrated levels. And the moderation bucket is an upper bound, as
above, because a not-yet-rescanned root lands in it.

The model scoring is a frozen input here: this notebook reads the resulting
summary rather than re-running a transformer, which is why the environment needs
no GPU stack.

In [ ]:
by_bucket = csv("moderation_vs_purge.csv", index_col=0)
print(by_bucket.round(4).to_string())
mod = float(by_bucket.loc["moderation (root survives)", "mean_tox"])
pur = float(by_bucket.loc["thread-purge (root deleted)", "mean_tox"])
print(f"\nmean-toxicity ratio moderation/purge: {mod / pur:.2f}x")

# The distribution panel needs the per-message scores, which are user message text
# and are not shipped. Re-scoring locally rebuilds `samp` with columns
# (bucket, tox); the summary above is what the thesis quotes.
SAMP_PARQUET = DATA / "moderation_vs_purge_sample.parquet"   # not shipped
samp = pd.read_parquet(SAMP_PARQUET)

order = list(by_bucket.index)
cols = {"moderation (root survives)": "#4c72b0", "thread-purge (root deleted)": "#dd8452"}

fig, axes = plt.subplots(2, 1, figsize=(5.48, 3.4), gridspec_kw={"height_ratios": [3, 1.3]})
for b in order:
    axes[0].hist(samp.loc[samp["bucket"] == b, "tox"], bins=40, density=True,
                 alpha=0.6, label=b.split(" (")[0], color=cols[b])
axes[0].set_yscale("log")
axes[0].set_xlabel("toxicity")
axes[0].set_ylabel("density (log)")
axes[0].set_title("Toxicity by deleted-comment bucket")
axes[0].legend()

ypos = np.arange(len(order)) * 0.4          # pack the thin bars close together
axes[1].barh(ypos, [by_bucket.loc[b, "mean_tox"] for b in order],
             height=0.27, color=[cols[b] for b in order])
axes[1].set_yticks(ypos)
axes[1].set_yticklabels([b.split(" (")[0] for b in order])
axes[1].set_ylim(ypos[-1] + 0.27, -0.27)    # snug box, first bucket on top
axes[1].set_xlabel("mean toxicity")
axes[1].set_title("Independent moderation vs thread-purge")
fig.tight_layout()
fig.savefig(FIGURES / "fig_6_11_moderation_vs_purge.png", dpi=150, bbox_inches="tight")
fig.savefig(FIGURES / "fig_6_11_moderation_vs_purge.pdf", bbox_inches="tight")
plt.show()

## Reading

The edit flag over-counts human editing by roughly an order of magnitude. That
matters beyond this corpus: `edit_date` is the only edit signal a snapshot corpus
has, so any study that reads it as an editing rate is reading mostly Telegram's
own housekeeping.

What edits actually do, on the diffable pairs, is change facts rather than tone.
The largest single type is cosmetic, and among the substantive ones additions and
removals dominate, with numbers being the most commonly churned element. Editing
is close to valence-neutral: it is not a mechanism for quietly softening or
hardening a message.

On deletions, comments are deleted several times more often than broadcast posts,
and most deleted comments went down with their thread rather than individually.
The independently moderated slice is the smaller one, and its contents are not
dramatically more toxic than the purged slice, which argues against reading
comment deletion as content moderation by default.